In [ ]:
# import libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
from numpy.random import seed
from tensorflow.random import set_seed
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf 
import math

# set random seed
seed(10)
set_seed(10)

training_dataset_name = "run53_mix_mega_shared"
testing_dataset_name = "run57_mix_mega_shared"

model_name = training_dataset_name+"_cart.keras"

val_number = 6000

# If not GPU is available, set the following to -1
os.environ["CUDA_VISIBLE_DEVICES"]="0"
number_of_detectors = 6
data_dir = "/data/test_newrepo/"
normalize= 1
plot=1


In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:

# load training dataset
file_path = data_dir+'/'+training_dataset_name+'_dataset.pkl'

with open(file_path, 'rb') as file:
    training_dataset_array = pickle.load(file)

# load testing dataset
file_path = data_dir+'/'+testing_dataset_name+'_dataset.pkl'

with open(file_path, 'rb') as file:
    testing_dataset_array = pickle.load(file)

In [ ]:

def diff_phi(a1, a2):
    # Calcola la differenza diretta
    diff = abs(a1 - a2)
    
    # Trova il percorso più breve tenendo conto del ciclo degli angoli
    if diff > 180:
        diff = 360 - diff
    
    return diff
    

def l2_normalize(data):
  """
  Normalize a NumPy array using the L2 (Euclidean) norm.

  Args:
    data (numpy.ndarray): The array to normalize. Can be a 1D vector
                         or a 2D matrix (where each row is a vector to normalize).

  Returns:
    numpy.ndarray: The L2-normalized array.
  """

  data = np.array(data)
  if data.ndim == 1:
    # Case: 1D vector
    norm = np.sqrt(np.sum(data**2))
    if norm == 0:
      return data  # Avoid division by zero if the vector is null
    return data / norm
  elif data.ndim == 2:
    # Case: 2D matrix (normalize each row)
    norms = np.sqrt(np.sum(data**2, axis=1, keepdims=True))
    # Handle the case of null norms (zero rows)
    norms[norms == 0] = 1
    return data / norms
  else:
    raise ValueError("Input must be a 1D or 2D array.")


def cartesian_to_spherical(x, y, z):
    # Calculate theta
    theta = np.degrees(np.arccos(z))
    
    # Calculate phi
    if x == 0 and y == 0:
        phi = 0
    else:
        phi = np.degrees(np.arctan2(y, x))
        if phi < 0:
            phi += 360
    
    return theta, phi
def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

def process_coordinates(coords_rad):

    # Separate the longitudes and latitudes in radians
    theta, phi = zip(*coords_rad)

    # Convert to Cartesian coordinates
    x = np.sin(np.radians(theta)) * np.cos(np.radians(phi))
    y = np.sin(np.radians(theta)) * np.sin(np.radians(phi))
    z = np.cos(np.radians(theta))
    
    return x,y,z

def process_coordinates_single(coords_rad):

    # Separate the longitudes and latitudes in radians
    theta, phi = coords_rad

    # Convert to Cartesian coordinates
    x = np.sin(np.radians(theta)) * np.cos(np.radians(phi))
    y = np.sin(np.radians(theta)) * np.sin(np.radians(phi))
    z = np.cos(np.radians(theta))
    
    return x,y,z


    
def prepare_dataset(data_array):

    dataset = np.empty((len(data_array),number_of_detectors))
    labels = np.empty((len(data_array),2))
    
    count = -1
    for element in data_array:
    
        count+=1
        
        dataset[count] = l2_normalize(element['counts'])
        labels[count][0] = float(element['coord'][0])
        labels[count][1] = float(element['coord'][1])
        
    labels = labels[:count+1]
    dataset = dataset[:count+1]

    # Convert to radians
    coords_rad = []
    
    for theta, phi in labels:
        #if lon > 90 and lon < 270:
            
        coords_rad.append([theta, phi])
        #else:
        #    continue

    x,y,z = process_coordinates(coords_rad)
    
    theta, phi = zip(*coords_rad)
  
    cartesian_labels = []

    for i in range(0,len(dataset)):
        cartesian_labels.append([x[i],y[i],z[i]])
        
        
    labels_norm = np.empty((len(dataset),3))
    count = -1
    for element in cartesian_labels:
        count+=1
        labels_norm[count][0] = element[0]
        labels_norm[count][1] = element[1]
        labels_norm[count][2] = element[2]
        
    
   
    return dataset, labels, labels_norm, theta, phi, coords_rad


In [ ]:
# prepare training and validation dataset
dataset, labels, labels_norm, theta, phi, coords_rad = prepare_dataset(training_dataset_array)

In [ ]:
# prepare testing and validation dataset
test_dataset, test_labels_raw, test_labels_norm, test_theta, test_phi, test_coords_rad = prepare_dataset(testing_dataset_array)

In [ ]:
dataset.shape

In [ ]:
test_dataset.shape

In [ ]:
#split val and test dataset
random_indices = np.random.permutation(len(dataset))

N = dataset.shape[0]
validation_indices = random_indices[:val_number]
train_indices = np.setdiff1d(np.arange(N), validation_indices)

validation_dataset = dataset[validation_indices]
training_dataset = dataset[train_indices]

validation_labels = labels_norm[validation_indices]
training_labels = labels_norm[train_indices]

test_labels = test_labels_norm


In [ ]:
plt.hist(theta,alpha=0.5)
plt.hist(test_theta,alpha=0.5)
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:
plt.hist(phi,alpha=0.5)
plt.hist(test_phi,alpha=0.5)
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:
plt.hist(training_labels[:,0],alpha=0.5)
plt.hist(test_labels[:,0],alpha=0.5)


In [ ]:
plt.hist(training_labels[:,1],alpha=0.5)
plt.hist(test_labels[:,1],alpha=0.5)


In [ ]:
plt.hist(training_labels[:,2],alpha=0.5)
plt.hist(test_labels[:,2],alpha=0.5)


In [ ]:
def custom_loss(y_true, y_pred):
    
    lambda_reg = 0.5
    
    #mse_loss = CosineSimilarity()(y_true, y_pred)  # Calcolo MSE tra coordinate previste e reali
    mse_loss = tf.keras.losses.MeanAbsoluteError()(y_true,y_pred)
    
    # Calcolo dell'errore rispetto all'equazione della sfera
    x_pred, y_pred, z_pred = tf.unstack(y_pred, axis=1)
    sphere_constraint_error = tf.square(x_pred**2 + y_pred**2 + z_pred**2 - 1)
    
    # Combinazione delle due loss con un peso per la regolarizzazione
    loss = mse_loss**2 + lambda_reg * sphere_constraint_error
    
    return loss

In [ ]:
# Define the model architecture
model = keras.Sequential([
    keras.layers.Dense(128*2, input_shape=(number_of_detectors,)), 
    keras.layers.LeakyReLU(alpha=0.1) ,
    keras.layers.Dropout(0.02),
    keras.layers.Dense(64*2), 
    keras.layers.LeakyReLU(alpha=0.1),
    keras.layers.Dropout(0.02),
    keras.layers.Dense(32*2),
    keras.layers.LeakyReLU(alpha=0.1),
    keras.layers.Dropout(0.02),
    keras.layers.Dense(3,activation="tanh")
])

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=custom_loss)
model.summary()



In [ ]:
# Define your learning rate schedule function
def variable_learning_rate(epoch, lr):
    if epoch < 10:
        return 0.01  
    elif epoch >= 4 and epoch < 50:
        return 0.001
    else:
        return  0.0001


# Define the LearningRateScheduler callback
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(variable_learning_rate, verbose=1)


history = model.fit(
    x=training_dataset,
    y=training_labels,
    epochs=2000,
    batch_size=256,
    validation_data=(validation_dataset,validation_labels),
    validation_split=0.2,
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, mode="min")
    ],
)


In [ ]:
#save model
save_model =False
if(save_model):
    model.save(data_dir+"/"+model_name)

In [ ]:
fig, ax1 = plt.subplots(1, 1,figsize=(10,5))
fig.suptitle('Training')
      
ax1.plot(history.history["loss"], label="Training Loss")
ax1.plot(history.history["val_loss"], label="Validation Loss")
    
ax1.legend()
plt.show() 

In [ ]:

# Evaluate testing dataset
pred_data = model.predict(test_dataset)

In [ ]:
pred_data_renorm = np.empty((len(pred_data),3))
test_data_renorm = np.empty((len(test_labels),3))

for j in range(0,len(pred_data)):
    pred_data_renorm[j][0]  = pred_data[j][0]
    pred_data_renorm[j][1] = pred_data[j][1]
    pred_data_renorm[j][2] = pred_data[j][2]

    
min_val = 0
max_val = 1

for j in range(0,len(test_labels)):
    test_data_renorm[j][0]  = test_labels[j][0]
    test_data_renorm[j][1] = test_labels[j][1]
    test_data_renorm[j][2] = test_labels[j][2]
    
pred_data_original = np.empty((len(pred_data),2))
test_labels_original = np.empty((len(test_labels),2))
        
for i, element in enumerate(test_data_renorm):

    theta,phi = cartesian_to_spherical(element[0], element[1], element[2])
    
    test_labels_original[i][0]=theta
    test_labels_original[i][1]=phi
    
for i, element in enumerate(pred_data_renorm):

    theta,phi = cartesian_to_spherical(element[0], element[1], element[2])
    
    pred_data_original[i][0]=theta
    pred_data_original[i][1]=phi
        
absolute_diffs_theta = np.abs(pred_data_original[:, 0] - test_labels_original[:, 0])
absolute_diffs_phi = np.abs(pred_data_original[:, 1] - test_labels_original[:, 1])

mae_theta = np.mean(absolute_diffs_theta)
mae_phi = np.mean(absolute_diffs_phi)

print("Mean Absolute Error for Theta:", mae_theta)
print("Mean Absolute Error for Phi:", mae_phi)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 0],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,0],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Theta")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 1],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,1],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Phi")
plt.legend()
plt.ylabel("Counts")

In [ ]:

# Esempio di utilizzo
theta1 = 90  # Latitudine del primo punto in gradi
phi1 = 350    # Longitudine del primo punto in gradi
theta2 = 90  # Latitudine del secondo punto in gradi
phi2 = 10    # Longitudine del secondo punto in gradi

distance = angular_distance(theta1, phi1, theta2, phi2)
print("Distanza angolare tra i due punti:", distance, "gradi")

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta sin")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,1],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 1],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta cos")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,2],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 2],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Phi sin")
plt.legend()
plt.ylabel("Counts")


In [ ]:
distances = []
theta_distances = []
phi_distances = []

for i in range(0,len(pred_data_original)):

    d = angular_distance(pred_data_original[i][0],pred_data_original[i][1],test_labels_original[i][0],test_labels_original[i][1])
    theta_dist = np.abs(pred_data_original[i][0]-test_labels_original[i][0])
    phi_dist = diff_phi(pred_data_original[i][1],test_labels_original[i][1])
    distances.append(d)
    theta_distances.append(theta_dist)
    phi_distances.append(phi_dist)



In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

nside = 32
npix = hp.nside2npix(nside)

m = np.array(distances)

hp.projview(
    m,
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    unit="cbar label",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="vertical",
    latitude_grid_spacing=30,
    projection_type="aitoff",
    title="Aitoff projection",
    cmap="turbo",
    nest=True
)

plt.show()


In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))

In [ ]:
# Definisci gli intervalli di 5 gradi
intervalli = np.arange(0, 185, 5)

# Raggruppa i conteggi in base agli intervalli di 5 gradi
conteggi_raggruppati = np.zeros((len(intervalli)))
conteggi = np.zeros((len(intervalli)))

tot_count = 0
for theta,phi,conteggio in zip(test_labels_original[:,0],test_labels_original[:,1], distances):
    
    if(True): #(phi>125 and phi<145) or 
        tot_count +=1
        indice_intervallo = int(theta // 5)
        conteggi_raggruppati[indice_intervallo] = conteggi_raggruppati[indice_intervallo]+conteggio
        conteggi[indice_intervallo]+=1

# Calcola la media dei conteggi in ciascun intervallo
conteggi_raggruppati = conteggi_raggruppati[:tot_count]
conteggi = conteggi[:tot_count]

# Visualizza l'istogramma
plt.figure(figsize=(10, 6))
plt.bar(intervalli[:-1], conteggi_raggruppati[:-1]/conteggi[:-1], width=5, align='edge',alpha=1,label="loc. error")
plt.xlabel('Theta (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error in function of theta')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
step = 10

# Definisci gli intervalli di 5 gradi
intervalli = np.arange(0, 365, step)

# Raggruppa i conteggi in base agli intervalli di 5 gradi
conteggi_raggruppati_1 = np.zeros((len(intervalli)))
conteggi_1 = np.zeros((len(intervalli)))

conteggi_raggruppati_2 = np.zeros((len(intervalli)))
conteggi_2 = np.zeros((len(intervalli)))

conteggi_raggruppati_3 = np.zeros((len(intervalli)))
conteggi_3 = np.zeros((len(intervalli)))

conteggi_raggruppati_4 = np.zeros((len(intervalli)))
conteggi_4 = np.zeros((len(intervalli)))

tot_count_1 = 0
tot_count_2 = 0
tot_count_3 = 0
tot_count_4 = 0

for theta,phi,conteggio in zip(test_labels_original[:,0],test_labels_original[:,1], distances):
    
    if(theta>20 and theta<55):
        tot_count_1 +=1
        indice_intervallo = int(phi // step)
        conteggi_raggruppati_1[indice_intervallo] = conteggi_raggruppati_1[indice_intervallo]+conteggio
        conteggi_1[indice_intervallo]+=1

    if(theta>55 and theta<150):
        tot_count_2 +=1
        indice_intervallo = int(phi // step)
        conteggi_raggruppati_2[indice_intervallo] = conteggi_raggruppati_2[indice_intervallo]+conteggio
        conteggi_2[indice_intervallo]+=1

    if(theta>150 and theta<180):
        tot_count_3 +=1
        indice_intervallo = int(phi // step)
        conteggi_raggruppati_3[indice_intervallo] = conteggi_raggruppati_3[indice_intervallo]+conteggio
        conteggi_3[indice_intervallo]+=1

# Calcola la media dei conteggi in ciascun intervallo
conteggi_raggruppati_1 = conteggi_raggruppati_1[:tot_count_1]
conteggi_1 = conteggi_1[:tot_count_1]

conteggi_raggruppati_2 = conteggi_raggruppati_2[:tot_count_2]
conteggi_2 = conteggi_2[:tot_count_2]

conteggi_raggruppati_3 = conteggi_raggruppati_3[:tot_count_3]
conteggi_3 = conteggi_3[:tot_count_3]

# Visualizza l'istogramma
plt.figure(figsize=(10, 6))
plt.bar(intervalli[:-1], conteggi_raggruppati_1[:-1]/conteggi_1[:-1], width=step, align='edge',alpha=0.5,label="loc. error theta=[20°,55°]")

plt.bar(intervalli[:-1], conteggi_raggruppati_2[:-1]/conteggi_2[:-1], width=step, align='edge',alpha=0.5,label="loc. error theta=[55°,150°]")

plt.bar(intervalli[:-1], conteggi_raggruppati_3[:-1]/conteggi_3[:-1], width=step, align='edge',alpha=0.5,label="loc. error theta=[150°,180°]")
plt.xlabel('Phi (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error in function of phi')
plt.grid(True)
plt.legend()
plt.show()